# Console-Based Chatbot using Hugging Face Transformers
This notebook demonstrates how to build a simple console-based chatbot using the `DialoGPT` model from Hugging Face.

## Step 1: Install Dependencies
First, we need to install the required libraries: `transformers` for the model and tokenizer, and `torch` for the PyTorch backend.

In [ ]:
# Install the required libraries for Hugging Face and PyTorch
!pip install transformers torch

## Step 2: Import Libraries and Load Model
We load the `DialoGPT-medium` model, which is fine-tuned for conversational responses. We also load its corresponding tokenizer.

In [ ]:
# Import necessary modules
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Define the model. DialoGPT is specifically fine-tuned for conversational responses.
model_name = "microsoft/DialoGPT-medium"

print("Downloading model and tokenizer... This may take a moment.")
# Load the pre-trained tokenizer and model from Hugging Face
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
print("Model loaded successfully!")

## Step 3: Chatbot Interaction Loop
The `while True` loop allows continuous interaction. We concatenate the new user input with the chat history so the model retains context of the conversation.

In [ ]:
# Start the chatbot interface
print("\nChatbot: Hello! I am your AI assistant. How can I help you today?")

# Initialize an empty variable to store the conversation history
chat_history_ids = None

# Create an infinite loop for continuous conversation
step = 0
while True:
    # 1. Accept User Input
    user_input = input("User: ")
    
    # 2. Check for Exit Condition
    if user_input.lower() in ['exit', 'quit']:
        print("Chatbot ends the conversation")
        break
        
    # 3. Encode the user input and append the end-of-string (EOS) token
    new_user_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')
    
    # 4. Maintain Conversation Flow
    # Append the new user input tokens to the chat history (if it exists)
    if step > 0:
        bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1)
    else:
        bot_input_ids = new_user_input_ids
        
    # Create attention mask to prevent generation warnings
    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)
    
    # 5. Generate Response
    # Generate a response taking the conversation history into account
    chat_history_ids = model.generate(
        bot_input_ids, 
        attention_mask=attention_mask,  # Suppresses the missing attention mask warning
        max_length=1000, 
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,         # Prevents the model from repeating the same phrases
        do_sample=True,                 # Enables slightly more creative/human-like responses
        top_k=50, 
        top_p=0.95,
        temperature=0.7
    )
    
    # 6. Display Output
    # Decode only the newly generated tokens (excluding the history) to text
    # Added clean_up_tokenization_spaces=False to suppress BPE tokenizer warning
    response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True, clean_up_tokenization_spaces=False)
    
    print(f"Chatbot: {response}")
    step += 1